In [1]:
# !wget https://data.ncl.ac.uk/ndownloader/articles/24574753/versions/1 -O ../data/Gutenberg.zip
# !mkdir ../data/Gutenberg
# !unzip ../data/Gutenberg.zip -d ../data/Gutenberg/
# !rm ../data/Gutenberg.zip

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [3]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=27 * 18, output_size=6):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [4]:
hyperparams = {
    "num_epochs": 256,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.1},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.001,
    "min_epochs_between_expansions": 16,
    "plateau_window_size": 5,
    "learning_rate": 4e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 1000,
    "weight_decay": 1e-3,
    "diff_threshold": 5e-3
}


In [5]:
train_x = np.load("../data/Gutenberg/train_x.npy")
train_y = np.load("../data/Gutenberg/train_y.npy")
valid_x = np.load("../data/Gutenberg/valid_x.npy")
valid_y = np.load("../data/Gutenberg/valid_y.npy")
test_x = np.load('../data/Gutenberg/test_x.npy')
test_y = np.load('../data/Gutenberg/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 27 * 18)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 27 * 18)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 27 * 18)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True, generator=g)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)

In [6]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [7]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
  api_key="frGbe20bUNVBiHdnVG40YiMwA",
  project_name="gutenberg",
  workspace="vanya9090"
)
experiment.log_parameters(hyperparams)
experiment.set_name("trash")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/vanya9090/gutenberg/fbdc454fadc7487f972ec2ba68d9ab21



In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, train_loader, val_loader, criterion,optimizer, hyperparams, device, experiment)

100%|██████████| 176/176 [00:01<00:00, 103.67it/s]


------------------------------
Epoch 1/256, Train Loss: 1.7300, Val Loss: 1.7006, Val Accuracy: 0.2898


100%|██████████| 176/176 [00:01<00:00, 137.28it/s]


------------------------------
Epoch 2/256, Train Loss: 1.6818, Val Loss: 1.6880, Val Accuracy: 0.2943


100%|██████████| 176/176 [00:01<00:00, 142.06it/s]


------------------------------
Epoch 3/256, Train Loss: 1.6713, Val Loss: 1.6844, Val Accuracy: 0.2986


... (136 outputs left out)

100%|██████████| 176/176 [00:02<00:00, 60.00it/s]


------------------------------
Epoch 71/256, Train Loss: 1.3721, Val Loss: 1.5832, Val Accuracy: 0.3664
avg_change: 0.002361320094628816
-0.0022014999931509573


 85%|████████▌ | 150/176 [00:02<00:00, 66.13it/s]


KeyboardInterrupt: 

In [ ]:
experiment.end()

In [ ]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [ ]:
accuracy, params

In [ ]:
torch.save(sparse_model, 'gutenberg.pt')